# B1.4 · What frontier SAST makes redundant, and what it doesn't

**Function B — Product & Application Security → The AppSec Engineer / Code Reviewer**  ·  *AI for Security*

---

**Risk.** Over-claiming replacement of pentest and business-logic testing.

**Control.** Honest scoping of where static reasoning substitutes and where it cannot.

**This lab.** Chart honestly what only SAST found, and what only DAST found.

| | |
|---|---|
| Open-source tooling | OWASP ZAP, OpenGrep |
| Open-weight models | Llama 3.3 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B1.4"))

Frontier SAST genuinely makes some work redundant. Being precise about *which* work is how you avoid both complacency and denial.

In [ ]:
from cybercommons import appsec

CATEGORIES = {
 "single-file pattern bugs (SQLi, command injection)":
    ("largely redundant", "high recall, and the pattern is local"),
 "cross-file taint through a framework":
    ("partly", "needs the call graph the model may not be given"),
 "business-logic authorization flaws":
    ("not redundant", "requires knowing what SHOULD be allowed"),
 "second-order and stored injection":
    ("not redundant", "source and sink are separated in time"),
 "secrets in source":
    ("fully redundant", "a regex was always enough — see scripts/check_secrets.py"),
 "design flaws (missing control entirely)":
    ("not redundant", "there is no code to point at"),
}
for cat, (verdict, why) in CATEGORIES.items():
    print(f"{verdict:20s} {cat}\n{'':20s} {why}\n")

Test the claim rather than accepting it: the scanner below finds the local bugs and is structurally unable to find the last category.

In [ ]:
found = appsec.scan_all()
print("local pattern bugs found:", sorted({f.cwe for f in found}))
print("missing-control findings:", "none — there is no line of code to match")

### Expect

The categorisation prints, and the scanner confirms it finds the local pattern classes while producing nothing for design-level gaps.

### Your turn

Take your last three real incidents. Which category was each? If most were 'not redundant', your SAST budget and your risk are pointed in different directions.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B1.4.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*